# Task 2: Logistic Regression Implementation

Train a logistic regression model and evaluate its performance using standard classification metrics.

**Goal**: Classify samples as cancer (1) or normal (0) and interpret the model components.

## 1. Initialize Project Environment

In [1]:
import logging
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)

print(f"Python {sys.version}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]


## 2. Define Configuration Parameters

In [2]:
@dataclass
class TaskConfig:
    handle: str
    artifacts_dir: Path = Path("artifacts")
    random_state: int = 42

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["artifacts_dir"] = str(info["artifacts_dir"])
        return info


CONFIG = TaskConfig(handle="rbals")
CONFIG.describe()

{'handle': 'rbals', 'artifacts_dir': 'artifacts', 'random_state': 42}

## 3. Implement Core Functionality

In [3]:
def run_logistic_regression(config: TaskConfig):
    # Load processed data
    X_train = pd.read_csv(config.artifacts_dir / "task1_X_train_scaled.csv")
    X_test = pd.read_csv(config.artifacts_dir / "task1_X_test_scaled.csv")
    y_train = pd.read_csv(config.artifacts_dir / "task1_y_train.csv").values.flatten()
    y_test = pd.read_csv(config.artifacts_dir / "task1_y_test.csv").values.flatten()

    # Train model
    model = LogisticRegression(random_state=config.random_state)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Performance metrics
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
    }

    logging.info(f"LogReg Metrics: {metrics}")

    # Interpret model: Logistic regression equation
    # P(y=1) = 1 / (1 + exp(-(W*X + b)))
    # Where W is weights and b is bias.
    weights = model.coef_[0]
    bias = model.intercept_[0]

    return metrics, weights, bias


METRICS, WEIGHTS, BIAS = run_logistic_regression(CONFIG)

2026-02-01 16:29:47,410 | INFO | LogReg Metrics: {'accuracy': 0.8529411764705882, 'precision': np.float64(0.8620689655172413), 'recall': np.float64(0.9615384615384616), 'f1_score': np.float64(0.9090909090909091)}


## 4. Validate with Unit Tests

In [4]:
assert METRICS["accuracy"] > 0.5, "Accuracy too low - model not learning"
assert len(WEIGHTS) > 0, "Weights not found"
print("[OK] Validation passed.")

[OK] Validation passed.


## 5. Export Results

In [5]:
EXPORT_DIR = CONFIG.artifacts_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save metrics to CSV
metrics_df = pd.DataFrame([METRICS])
metrics_out = EXPORT_DIR / "task2_logistic_regression_metrics.csv"
metrics_df.to_csv(metrics_out, index=False)

# Save weights to CSV (Top 10 features for interpretation)
weights_df = pd.DataFrame(
    {
        "Feature": pd.read_csv(EXPORT_DIR / "task1_X_train_scaled.csv").columns,
        "Weight": WEIGHTS,
    }
)
weights_df = weights_df.sort_values(by="Weight", ascending=False)
weights_out = EXPORT_DIR / "task2_logistic_regression_weights.csv"
weights_df.to_csv(weights_out, index=False)

print(f"[OK] Metrics and Weights saved to artifacts/")

[OK] Metrics and Weights saved to artifacts/
